<img src="logo.png" alt="Vegeta" width="240">

# PEREGRINE — a falcon-inspired folding-wing farm drone (28)

PEREGRINE is a light, cheap, electric fixed-wing that keeps wild animals and pest bird flocks off the crops **with the
noise of its propeller** — the sound does the job, so there is no acoustics here. It patrols the field, stoops like a
falcon at a deer, flies between the trees and the houses of the countryside, follows and herds a flock of starlings
off the crop, rests on the farmhouse roof between sorties and starts again by hanging on its propeller (or from a hand
throw).

**The scientific question.** Its wings can sweep back in flight — the falcon's tuck. *Is the folding wing worth what
it costs to build?* So this notebook designs **three aircraft on one body** — the same fuselage, tail, motor,
propeller, battery and electronics; only the wing differs — and flies all three through the same tasks:

| | wing | what it stands for |
|---|---|---|
| **A** | large fixed wing (1.1 m) | best cruise; the cheapest |
| **B** | small fixed wing (0.8 m, falcon-like wing loading) | the cheap way to be fast and agile |
| **C** | A's wing on two hinges, sweeping back 0–60° in flight, 85° stowed on the roof | A's cruise *and* B's agility? |

**C has to beat both.** If it only beats A, the answer is "build B".

Nothing is designed twice: the body is MERLIN's tractor fuselage (notebook 26, `designs/merlin.py`), the wing sections,
the tail and the wing loft are notebook 09's `FixedWing`, the propeller is notebook 25's library, the flight models build
on MERLIN's `Airframe`, `Unit` and point-mass flight, the actuators are `designs/actuators.py`. New:
`designs/peregrine.py` (the hinged wing, the hinge lug, the planform, the folded drag build-up) and
`designs/peregrine_flight.py` (a vortex lattice for the folded wing, stability and trim, the stoop, gaps, the roof,
the hinge, the flock, the sortie and the movie, the decision table).

```
Part 1 — the aircraft    1 mission → 2 mass & cost → 3 CAD, the hinge station → 4 aerodynamics vs fold
                         → 5 stability & trim → 6 propulsion
Part 2 — the flight      7 envelopes → 8 on par with birds → 9 the stoop → 10 trees and houses → 11 herding a flock
                         → 12 the roof → 13 structure & the hinge → 14 printing → 15 the sortie & the movie
Part 3 — the verdict     16 does folding pay? → 17 export
```

The OpenFOAM cell runs when OpenFOAM is there (`VEGETA_SKIP_OPENFOAM=1` skips it; the drag build-up is the polar then);
`PEREGRINE_QUICK=1` runs fewer flocks and a shorter movie.

In [ ]:
import dataclasses, json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from vegeta import dedalus, talos, aeromant, mellonia, core
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
sys.path.insert(0, "designs")
import peregrine as pg
import peregrine_flight as pf
import actuators as act

ROOT = Path("_runs/peregrine"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
OUT = Path("output") / "28_peregrine"; OUT.mkdir(parents=True, exist_ok=True)       # every movie of this notebook
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("PEREGRINE_QUICK") == "1"
KEYS = ("A", "B", "C")
NAME = pg.VARIANT_NAME
COLOR = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#d62728"}          # the folding wing in red
FOLDS = (0.0, 15.0, 30.0, 45.0, 60.0)                             # C's folds in flight
TUCK, STOW = pg.FOLD_TUCK, pg.FOLD_STOW
pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
print(f"CFD {'on if OpenFOAM is found' if RUN_CFD else 'skipped'}; {'QUICK check' if QUICK else 'full runs'}")

# Part 1 — the aircraft

## 1. Mission and requirements — edit here

An abstract farm (`pf.Farm`): a 4.8 ha crop field, the farmhouse and the barn west of it, two tree lines with gaps
(the hedges and windbreaks), the roof perch on the farmhouse ridge (8.5 m), and the wood 400 m north where the birds
are pushed to. One sortie: start from the ridge → a patrol lap of the field at the slowest economical speed (the noise
on the crop) → a stoop at a deer and a low pass over it at full power → a starling flock arrives over the crop and is
herded to the wood → back to the ridge.

The targets are the questions the notebook asks of each wing; the birds come in section 8.

In [ ]:
FARM = pf.Farm()
MISSION = dict(patrol_agl_m=25.0, stoop_from_m=60.0, stoop_high_m=150.0, pass_agl_m=3.0, flock="starling",
               battery_wh=24.4, usable_fraction=0.80, n_struct=5.0, deer_xy=(200.0, -60.0))
TARGETS = pd.DataFrame({
    "target": ["≥ 45 min patrol on one 3S 2200 mAh pack", "≥ 1.3 x the pest birds' escape speed", "turn inside a bird's radius (~10 m)",
               "pass a 1.2 m gap in a tree line", "stoop from 60 m and pull out at 3 m", "herd a starling flock off the crop",
               "start from a roof ridge (prop-hang or hand throw)", "stay on the ridge in a breeze", "cheap: calculated, not capped"],
    "why": ["a morning's patrol in two or three sorties", "to follow and push a flock", "to stay with a turning flock",
            "hedges, windbreaks, between farm buildings", "the falcon's way to scare ground animals", "the main job in autumn",
            "no runway on a farm", "it rests there between sorties", "a fleet per farm; a lab first"]})
display(TARGETS)

fig, ax = plt.subplots(figsize=(9, 9))
x0, y0, x1, y1 = FARM.field_xy
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, color="#e2c35a", label=f"crop field ({(x1 - x0) * (y1 - y0) / 1e4:.1f} ha)"))
for (bx, by, L, Wd, e, r) in FARM.buildings:
    ax.add_patch(plt.Rectangle((bx - L / 2, by - Wd / 2), L, Wd, color="#b0745a"))
tr = FARM.trees()
ax.scatter(tr[:, 0], tr[:, 1], s=tr[:, 3] ** 2 * 2, c="#2f6b33", label="tree lines (with gaps)")
ax.plot(*FARM.perch[:2], "k^", ms=10, label="roof perch (ridge, 8.5 m)")
ax.add_patch(plt.Circle(FARM.safe_xy, FARM.safe_r, fill=False, color="#2e6b2e", lw=2, label="the wood (safe area)"))
ax.plot(*MISSION["deer_xy"], marker="^", color="#8b5a2b", ms=10, ls="", label="a deer")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=8); ax.set(xlabel="x east [m]", ylabel="y north [m]", title="the farm")

## 2. Mass and cost budget

The bill of materials (`pf.BOM_ITEMS`): hobby-market parts with their mass and a typical 2026 online price (single
units; estimates — edit them), placed along the fuselage for the centre of gravity. The printed airframe's mass is its
wetted skin area from the CAD's profiles at MERLIN's 0.7 kg/m² (LW-PLA shells, ribs, glue), priced as filament. The
battery is placed to give a 10 % static margin with the wing spread (as builders do it, section 5).

C adds the fold hardware: two fold servos (assumed here; section 13 sizes them from the hinge loads), hinge pins with
four flanged bearings, four printed lugs (section 14 weighs them), the fold linkage, the spar's split sleeves and the
extra wiring. **The BOM is calculated, not capped.**

In [ ]:
FOLD_SERVO = "micro-metal 20 g"      # assumed here, sized in section 13
LUG_G = 4.0                          # g per printed lug, weighed by the slicer in section 14


def budget(fold_servo=FOLD_SERVO, lug_g=LUG_G):
    boms = {k: pf.bom({"variant": k}, fold_servo=fold_servo if k == "C" else None, printed_lug_g=lug_g) for k in KEYS}
    table = pd.concat({k: b for k, b in boms.items()}, axis=1).fillna(0.0)
    return boms, table


BOMS, BOM_TABLE = budget()
MASS = {k: BOMS[k]["mass [g]"].sum() / 1000 for k in KEYS}
display(BOM_TABLE.round(1))
SUMMARY = pd.DataFrame({NAME[k]: {"all-up mass [g]": BOMS[k]["mass [g]"].sum(), "parts cost [EUR]": BOMS[k]["EUR"].sum(),
                                  "items": int((BOMS[k]["mass [g]"] > 0).sum())} for k in KEYS})
display(SUMMARY.round(0))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
groups = {"airframe": "airframe", "battery": "battery", "motor|ESC|propeller": "drive", "fold|hinge|lug|split": "fold hardware"}
for i, k in enumerate(KEYS):
    b = BOMS[k]
    bottom_m = bottom_e = 0.0
    for pat, lab in list(groups.items()) + [(None, "electronics, servos, rest")]:
        sel = b.index.str.contains(pat) if pat else ~np.any([b.index.str.contains(q) for q in groups], axis=0)
        m_, e_ = b.loc[sel, "mass [g]"].sum(), b.loc[sel, "EUR"].sum()
        c = {"airframe": "#8d6e63", "battery": "#546e7a", "drive": "#26a69a", "fold hardware": "#d62728"}.get(lab, "#bdbdbd")
        ax[0].bar(i, m_, bottom=bottom_m, color=c, label=lab if i == 2 else None); bottom_m += m_
        ax[1].bar(i, e_, bottom=bottom_e, color=c); bottom_e += e_
for a, yl in zip(ax, ("g", "EUR")):
    a.set_xticks(range(3)); a.set_xticklabels([NAME[k] for k in KEYS], fontsize=8); a.set_ylabel(yl); a.grid(alpha=0.3, axis="y")
ax[0].set_title("mass"); ax[1].set_title("parts cost"); ax[0].legend(fontsize=8)
fig.tight_layout()

## 3. The three wings (Dedalus) — and where the hinge can be

`Peregrine` is MERLIN's tractor body with `FixedWing`'s wing; `wing_hinge="root"` puts each outer panel on a vertical
hinge on the spar, and `fold_deg` rotates it aft about that hinge (the knuckle over the pin stands 2 mm proud of the
skins). The design file is copied into the run folder, as in 26, so a copilot proposal edits the copy.

**A finding before any flight.** The plan was a falcon-like tuck *at the wing root*. Behind its hinge the panel's root
chord (200 mm of the 250 mm root chord) swings inboard as the wing folds — through the body and into the other wing.
`pg.fold_limit` gives the largest fold before the panels collide: with the hinge at the root it is under 10°. A
falcon does not tuck at the shoulder either: the hand wing sweeps back at the wrist and the elbow while the arm wing
stays. So C's hinge sits at **30 % of the half span** (`hinge_y_frac`): a fixed inner wing (the arm) and a folding
outer panel (the hand) that slides over the inner wing and the body — 60° in the stoop, 85° stowed over the back.

In [ ]:
design_file = ROOT / "peregrine.py"
shutil.copy(Path("designs/peregrine.py"), design_file)
pdesign = dedalus.load_design(f"{design_file}:Peregrine")
display(pd.DataFrame(pdesign.params.table()).set_index("name"))
GEOM = {}
for k in KEYS:
    for f in ((0.0, 30.0, TUCK, STOW) if k == "C" else (0.0,)):
        GEOM[(k, f)] = g = pdesign.generate(**pg.VARIANTS[k], fold_deg=f)
        print(f"{NAME[k]:<22s} fold {f:4.0f}°: valid {g.shape.isValid()}, {len(g.shape.Solids())} solid, "
              f"box {np.round(g.dimensions, 0)} mm, wetted {g.surface_area / 1e6:.3f} m²")

fracs = np.linspace(0.0, 0.6, 31)
lim = [pg.fold_limit({"variant": "C", "hinge_y_frac": f})["max_fold_deg"] for f in fracs]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1.6]})
ax[0].plot(fracs, lim, color=COLOR["C"]); ax[0].axhline(TUCK, ls="--", color="k", lw=0.8, label="stoop tuck 60°")
ax[0].axhline(STOW, ls=":", color="k", lw=0.8, label="roof stow 85°")
ax[0].axvline(pg.resolve({"variant": "C"})["hinge_y_frac"], color="grey", lw=1, label="the design: 30 %")
ax[0].set(xlabel="hinge station (0 = the wing root, 1 = the tip)", ylabel="largest fold before the panels collide [deg]",
          title="where the hinge can be"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for f, a_ in zip((0.0, 30.0, TUCK, STOW), (1.0, 0.55, 0.4, 0.25)):
    for poly in pg.outline({"variant": "C"}, fold_deg=f)["top"]:
        ax[1].fill(poly[:, 1], -poly[:, 0], color=COLOR["C"], alpha=a_ * 0.5, ec="k", lw=0.4)
for poly in pg.outline({"variant": "B"})["top"]:
    ax[1].plot(np.r_[poly[:, 1], poly[0, 1]] + 1.4, -np.r_[poly[:, 0], poly[0, 0]], color=COLOR["B"], lw=1)
ax[1].text(1.4, 0.25, "B", color=COLOR["B"], ha="center"); ax[1].text(0, 0.25, "C: 0 / 30 / 60 / 85°", color=COLOR["C"], ha="center")
ax[1].set_aspect("equal"); ax[1].set(xlabel="y [m]", ylabel="-x [m] (nose up)", title="top view: C folding, and B"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
try:
    for key in (("A", 0.0), ("C", TUCK), ("C", STOW)):
        dviz.show(dviz.plot3d(GEOM[key]))
except ImportError as e:
    print("3D view needs pyvista:", e)
fig, ax = plt.subplots(figsize=(13, 3))
for poly in pg.outline({"variant": "C"})["side"]:
    ax.fill(poly[:, 0], poly[:, 1], color=COLOR["C"], alpha=0.35, ec="k", lw=0.6)
ax.axvline(pg.outline({"variant": "C"})["propulsor_x"], color="k", ls=":", lw=1)
ax.set_aspect("equal"); ax.set(xlabel="x [m]", title="side view (the propeller plane dotted)"); ax.grid(alpha=0.3)

## 4. Aerodynamics against the fold

Nothing in the repository did swept wings, so `pf.vlm` is a vortex lattice: horseshoe vortices on `pg.planform`'s flat
panels, meshed in **streamwise strips** (a folded panel's own chords are not streamwise; its trailing vortices must
still run with the flow), Kutta–Joukowski forces, induced drag in the Trefftz plane. The folded panel is a lifting
surface only outboard of the hinge station — inboard it lies over the inner wing. Its checks (in
`designs/tests/test_peregrine.py`): a rectangular wing's lift slope within 8 % of Helmbold, span efficiency ~0.95–1,
neutral point at the quarter chord, lift falling smoothly as a panel rotates.

`pf.aero` gives, with the tail: CLα, the induced-drag factor `k = CDi/CL²` (the number that sets cruise power), the
neutral point (plus a fuselage term, Raymer); `pf.cl_max` the maximum lift (the visible panel area × cos fold); the drag
build-up (`pg.drag_buildup`, MERLIN's Raymer method on the folded geometry: the hidden skin is not wetted, each hinge
costs 3 % of the wing's profile drag).

In [ ]:
POLAR = {k: pf.fold_polar({"variant": k}, (0.0,)) for k in "AB"}
POLAR["C"] = pf.fold_polar({"variant": "C"}, FOLDS + (70.0, STOW))
display(pd.concat({k: v for k, v in POLAR.items()}).round(4))
cols = ["CLα [1/rad]", "k = CDi/CL²", "x_np [m]", "CL_max", "Cd0·S [cm²]", "span [m]"]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for a, c in zip(ax.flat, cols):
    a.plot(POLAR["C"].index, POLAR["C"][c], "o-", color=COLOR["C"], label=NAME["C"])
    for k in "AB":
        a.axhline(POLAR[k][c].iloc[0], color=COLOR[k], ls="--", label=NAME[k])
    a.axvspan(TUCK, STOW + 2, color="grey", alpha=0.1)
    a.set(xlabel="fold [deg]", title=c); a.grid(alpha=0.3)
ax[0, 0].legend(fontsize=8)
fig.suptitle("the folded wing against the two fixed ones (grey: beyond the stoop tuck — the roof stow)")
fig.tight_layout()

**Reading it.** Folding keeps the planform area but loses span: the lift slope falls (a gust lifts less — the
structure sees less in a dive), the induced-drag factor rises steeply (slow flight costs more), the maximum lift falls
(the stall speed rises), and the neutral point moves aft (more stable). The parasite drag falls only by the skin that
hides over the inner wing — **a sweep tuck does not make the aircraft much slicker**, unlike the falcon, which also
presses its wings against its body. B, with less area and span, has a higher `k` and less drag area than A.

The CFD check (optional): one coarse RANS of C spread and tucked at 2° (09a's case), replacing the build-up's Cd0 through
the parabolic polar where it ran.

In [ ]:
CFD_CD0 = {}
env = None
if RUN_CFD:
    try:
        env = aeromant.OpenFOAMEnvironment.detect()
    except Exception as e:
        print("no OpenFOAM here:", e)
if env is not None:
    ws_cfd = core.Workspace.create(ROOT / "cfd", name="PEREGRINE CFD")
    d_cfd = ws_cfd.add_design("peregrine", f"{design_file}:Peregrine")
    S_C = POLAR["C"]["span [m]"].iloc[0]
    for f in (0.0, TUCK):
        rev = d_cfd.new_revision(**pg.VARIANTS["C"], fold_deg=f, angle_of_attack_deg=2.0, note=f"C fold {f:.0f}, 2 deg")
        rev.generate(stl_tolerance=0.2)
        pl = pg.planform({"variant": "C"}, f)

        def case(rev, workdir, pl=pl):
            return aeromant.CFDCase("rans_ksst_external", rev.stl,
                dict(velocity=20.0, kinematic_viscosity=1.5e-5, density=1.225, reference_area=pl["S_ref"], reference_length=pl["c_ref"],
                     center_of_rotation=(0.07, 0.0, 0.0), iterations=400, residual_target=1e-4, surface_level=4, near_level=3,
                     wake_level=2, cells_per_length=2.0), workdir=workdir, geometry_units="mm", environment=env)
        ev = rev.run_cfd(f"fold_{f:.0f}", case, progress=True)
        if ev.ok:
            a_ = pf.aero({"variant": "C"}, f)
            CFD_CD0[f] = ev.metrics["Cd"] - a_["k"] * ev.metrics["Cl"] ** 2
            print(f"fold {f:.0f}: Cl {ev.metrics['Cl']:.3f}, Cd {ev.metrics['Cd']:.4f} -> Cd0 {CFD_CD0[f]:.4f} "
                  f"(build-up {POLAR['C'].loc[f, 'Cd0']:.4f})")
else:
    print("CFD not run: the drag build-up is the polar")

## 5. Stability and trim against the fold

The centre of gravity from the mass table (the outer panels' skin, spar and aileron servos move aft with the fold), the
neutral point from the lattice, the static margin, and the trim — angle of attack and elevator — at three lift
coefficients: the slow patrol, the stoop (n ≈ 0.5 at 25 m/s) and the pull-out (5 g at 30 m/s). The battery stays where
section 2 put it (10 % margin spread). The question: does the tuck make C so stable that the elevator cannot pull it
out of the dive?

In [ ]:
BATTERY_X = {k: pf.mass_table({"variant": k}).filter(like="battery", axis=0)["x [m]"].iloc[0] for k in KEYS}
rows = []
for k in KEYS:
    for f in ((0.0,) if k != "C" else FOLDS + (STOW,)):
        t = pf.mass_table({"variant": k}, fold_deg=f, fold_servo=FOLD_SERVO, battery_x=BATTERY_X[k])
        S = pf.aero({"variant": k}, f)["S_ref"]
        for case, V, n in (("patrol 10 m/s", 10.0, 1.0), ("stoop 25 m/s, n 0.5", 25.0, 0.5), ("pull-out 30 m/s, 5 g", 30.0, 5.0)):
            cl = n * t["mass [kg]"].sum() * pf.G / (0.5 * pf.RHO * V ** 2 * S)
            st = pf.stability({"variant": k}, f, t, cl=cl)
            rows.append({"wing": k, "fold [deg]": f, "case": case, **{c: st[c] for c in ("x_cg [m]", "x_np [m]", "static margin [%MAC]",
                         "CL trim", "alpha trim [deg]", "elevator trim [deg]")}})
STAB = pd.DataFrame(rows)
display(STAB.pivot_table(index=["wing", "fold [deg]"], columns="case", values=["static margin [%MAC]", "elevator trim [deg]"]).round(1))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
c_ = STAB[(STAB.wing == "C")]
ax[0].plot(c_[c_.case.str.startswith("patrol")]["fold [deg]"], c_[c_.case.str.startswith("patrol")]["static margin [%MAC]"], "o-", color=COLOR["C"])
ax[0].set(xlabel="fold [deg]", ylabel="% MAC", title="C: static margin"); ax[0].grid(alpha=0.3)
for case, ls in zip(c_.case.unique(), ("-", "--", ":")):
    s_ = c_[c_.case == case]
    ax[1].plot(s_["fold [deg]"], s_["elevator trim [deg]"], ls, marker="o", color=COLOR["C"], label=case)
ax[1].axhspan(-20, 20, color="green", alpha=0.07, label="elevator travel ±20°")
ax[1].set(xlabel="fold [deg]", ylabel="deg (negative: trailing edge up)", title="C: elevator to trim"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()

**Reading it.** The tuck moves the neutral point aft faster than the folded panels move the centre of gravity: the
static margin grows from 10 % to about 30 % at 45–60° and falls back near the stow (the stowed panels hardly lift). In the
stoop and the pull-out the elevator trims C within about ±10° at every fold — **the tuck does not lock it into the
dive**. Slow and tucked it needs about −21°, beyond a ±20° elevator: C must not tuck at patrol speed — the fold is a
high-speed configuration (tucked at 60° it stalls at ~10 m/s anyway, section 7).

## 6. Propulsion

One tractor drive for all three wings from notebook 25's propeller library (`pf.drive`: `propulsor_maps.unit` with a
tractor installation), limited to 300 W electrical (a 30 A ESC on 3S), plus 8 W drawn whatever the thrust (the motor's
no-load losses and the avionics — at a 10 W cruise that is not negligible). Requirements: static thrust well above the
weight of the heaviest wing (C) to **hang on the propeller** on the roof, and a top speed well above the birds'.

In [ ]:
import propulsor_maps as pm
LIB = pm.load()
cands = []
for pid in ("prop-9x5-b2", "prop-9x7-b2", "prop-9x9-b2"):
    u = pf.drive(pid, 300.0, library=LIB)
    ac_tmp = pf.make_aircraft("C", {"variant": "C"}, MASS["C"])
    e = pf.envelope(ac_tmp, u)
    cands.append({"propeller": pid, "static thrust [N]": u.full(0.0)[0], "T/W (C)": u.full(0.0)[0] / (MASS["C"] * pf.G),
                  "hover power (C) [W]": u.electrical_power(0.0, MASS["C"] * pf.G), "top speed C [m/s]": e["v_top"],
                  "endurance C [min]": e["endurance_min"]})
display(pd.DataFrame(cands).set_index("propeller").round(2))
UNIT = pf.drive("prop-9x5-b2", 300.0, library=LIB)
print("chosen:", UNIT.name, "— the most static thrust (the prop-hang) at a top speed still far above the birds'")

ACS = {k: pf.make_aircraft(k, {"variant": k}, MASS[k], folds=FOLDS if k == "C" else (0.0,), n_struct=MISSION["n_struct"],
                           cd0_correction=0.0) for k in KEYS}
if CFD_CD0:
    for f, af in ACS["C"].airframes.items():
        near = min(CFD_CD0, key=lambda q: abs(q - f))
        af.cd0 += CFD_CD0[near] - POLAR["C"].loc[near, "Cd0"]
V = np.linspace(0, 45, 91)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(V, [UNIT.full(v)[0] for v in V], "k", lw=2, label="full thrust")
for k in KEYS:
    ax.plot(V[V > 6], ACS[k].af(0).drag(V[V > 6]), color=COLOR[k], label=f"drag {NAME[k]}")
ax.plot(V[V > 8], ACS["C"].af(TUCK).drag(V[V > 8]), color=COLOR["C"], ls="--", label="drag C tucked 60°")
for k in KEYS:
    ax.axhline(MASS[k] * pf.G, color=COLOR[k], ls=":", lw=0.8)
ax.set(xlabel="airspeed [m/s]", ylabel="N", title="thrust and drag (dotted: the weights — the prop-hang needs thrust above them at 0 m/s)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Part 2 — the flight

## 7. Envelopes

MERLIN's speed polar (`merlin_flight.performance`) per wing and fold, with the endurance at the speed of least power,
the range, the best glide, the tightest turns (instantaneous: lift- or structure-limited at 5 g; sustained: full thrust
= drag) and the roll rate at full aileron (`p b / 2V = 0.09`, the ailerons' effectiveness falling as cos fold).

In [ ]:
ENV = pf.envelope_table(list(ACS.values()), UNIT, {"C": [0.0, 30.0, TUCK]}, battery_wh=MISSION["battery_wh"],
                        usable=MISSION["usable_fraction"])
display(ENV.round(2))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k, f, ls in (("A", 0, "-"), ("B", 0, "-"), ("C", 0, "-"), ("C", 30, "--"), ("C", TUCK, ":")):
    e = pf.envelope(ACS[k], UNIT, fold=f)
    lab = f"{NAME[k]}" + (f", fold {f:.0f}°" if k == "C" else "")
    ax[0].plot(e["V"], e["P_level"], ls, color=COLOR[k], label=lab)
    ax[1].plot(e["V"], e["radius_inst"], ls, color=COLOR[k], label=lab)
ax[0].set(xlabel="airspeed [m/s]", ylabel="W", title="electrical power in level flight", ylim=(0, 300)); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="airspeed [m/s]", ylabel="m", title="tightest turn radius (lift or 5 g)", ylim=(0, 40)); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 8. On par with birds?

`pf.BIRDS` places the pests and the peregrine itself (order-of-magnitude literature values, cited in the module): their
cruise and escape speeds and the load factor of their hard turns, so the turn radius `V²/(g√(n²−1))`. The drones'
curves are their tightest turns against speed (section 7). To follow and push a flock the drone needs a speed margin
over the birds' escape speed and a turn radius no larger than theirs at that speed.

In [ ]:
BIRD = pf.bird_table()
display(BIRD.round(1))
fig, ax = plt.subplots(figsize=(10, 6))
for k, f, ls in (("A", 0, "-"), ("B", 0, "-"), ("C", 0, "-"), ("C", TUCK, ":")):
    e = pf.envelope(ACS[k], UNIT, fold=f)
    m_ = e["V"] <= e["v_top"]
    ax.plot(e["V"][m_], e["radius_inst"][m_], ls, color=COLOR[k], label=NAME[k] + (f" tucked {f:.0f}°" if f else "") + f" (to its top speed {e['v_top']:.0f} m/s)")
for name, (m, vc, vx, n, vs) in pf.BIRDS.items():
    s = math.sqrt(n ** 2 - 1)
    ax.plot([vc, vx], [vc ** 2 / (pf.G * s), vx ** 2 / (pf.G * s)], "o-", color="k", alpha=0.6)
    ax.annotate(name, (vx, vx ** 2 / (pf.G * s)), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set(xlabel="speed [m/s]", ylabel="turn radius [m]", yscale="log", title="agility: the drones' tightest turns and the birds' (cruise → escape)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
margin = {NAME[k]: ENV.loc["top speed [m/s]"].filter(like=k).max() / max(v[2] for v in pf.BIRDS.values() if v[4] is None) for k in KEYS}
print("top speed / fastest pest's escape speed:", {k: round(v, 2) for k, v in margin.items()})

## 9. The stoop

`pf.stoop`: from level flight a push-over onto the line through the target, the dive with the propeller stopped
(the falcon's way — and a silent approach, the noise switched on at the pass), the pull-out at the most the lift and the
5 g structure allow, to level flight 3 m over the deer. C dives tucked at 60° and unfolds at the servo's rate as it pulls
out; the pull-out height is found so the bottom of the curve is at 3 m. Two dives: from 60 m (the patrol, shallow) and
from 150 m (steep, 63°).

In [ ]:
STOOP = {}
rows = []
for k, tuck in (("A", 0.0), ("B", 0.0), ("C", 0.0), ("C", TUCK)):
    for h0, xt in ((MISSION["stoop_from_m"], 90.0), (MISSION["stoop_high_m"], 75.0)):
        s = pf.stoop(ACS[k], UNIT, h0=h0, x_target=xt, h_pass=MISSION["pass_agl_m"], tuck=tuck)
        STOOP[(k, tuck, h0)] = s
        rows.append({"wing": NAME[k] + (" tucked" if tuck else ""), "from [m]": h0, "dive angle [deg]": s["dive_angle_deg"],
                     "time to the target [s]": s["time_s"], "top speed [m/s]": s["V_max"], "speed over it [m/s]": s["V_bottom"],
                     "pull-out from [m]": s["h_pull"], "peak n": s["n_peak"]})
display(pd.DataFrame(rows).set_index(["wing", "from [m]"]).round(2))
print("terminal speeds (vertical, zero lift):", {NAME[k]: round(pf.terminal_speed(ACS[k].af(0)), 1) for k in KEYS},
      "| C tucked:", round(pf.terminal_speed(ACS["C"].af(TUCK)), 1), "m/s")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for (k, tuck, h0), s in STOOP.items():
    if h0 != MISSION["stoop_high_m"]:
        continue
    ls = ":" if tuck else "-"
    ax[0].plot(s["x"], s["h"], ls, color=COLOR[k], label=NAME[k] + (" tucked" if tuck else ""))
    ax[1].plot(s["t"], s["V"], ls, color=COLOR[k])
ax[0].set(xlabel="m", ylabel="m", title=f"the stoop from {MISSION['stoop_high_m']:.0f} m"); ax[0].set_aspect("equal"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="s", ylabel="m/s", title="airspeed"); ax[1].grid(alpha=0.3)
fig.tight_layout()

**Reading it.** At these heights the dive is far from terminal speed: all four reach 41–43 m/s from 150 m and take
6.6–6.7 s to the deer. The tuck adds about 1 m/s and saves a few hundredths of a second over C spread — **in the stoop
the fold buys almost nothing**, because a sweep tuck keeps the area (a falcon also presses its wings onto its body).
B, with less wing, dives as fast as tucked C.

## 10. Between trees and houses

`pf.gap_passage`: the narrowest gap a wing passes, 3 m deep (a hedge or a tree line) or 10 m (the yard between two
houses), losing at most 1 m of height and with 15 cm tracking margin each side. A fixed wing **banks through** (rolls
at its roll rate, holds the bank with the lift at `W cos φ` so the path stays straight, sinking at `g sin² φ`); C may also
fold (up to the 60° tuck, at ≥ 1.25 × its folded stall speed) and bank on top. The U-turn corridor is the width a
reversal needs: `2 r_min + b`.

In [ ]:
rows = {}
for k in KEYS:
    r = {}
    for d in (3.0, 10.0):
        g = pf.gap_passage(ACS[k], depth=d)
        r[f"gap {d:.0f} m deep [m]"] = g["width_m"]
        r[f"how ({d:.0f} m)"] = f"fold {g['fold_deg']:.0f}°, bank {g['bank_deg']:.0f}° at {g['speed']:.1f} m/s"
    e = pf.envelope(ACS[k], UNIT)
    r["U-turn corridor [m]"] = 2 * e["r_min_inst"] + ACS[k].span(0)
    rows[NAME[k]] = r
GAPS = pd.DataFrame(rows)
display(GAPS)

**Reading it.** The large fixed wing needs ~1.3 m for a hedge (it can bank only so far before it sinks a metre); B
banks steeper through ~1.0 m; C folds to 60° *and* banks to pass ~1.0 m — the same as B. A U-turn is tightest for A
(lowest wing loading), then C (heavier), then B.

## 11. Herding a flock

`pf.Flock` is a boids flock (cohesion, alignment, separation), pulled to the crop where it wants to feed and circle,
and afraid of the drone within 45 m (after Paranjape et al. 2018, who herded real bird flocks with a drone). `pf.herd`
aims the drone at a point behind the flock on the far side from the wood and flies there at its herding speed — a fixed
wing cannot hover behind a flock, so it circles that point fast; C tucks 30° on straight runs. Done when 90 % of the birds
are 60 m outside the field. Starlings and rooks, three herding speeds, several random flocks each.

In [ ]:
SEEDS = range(2 if QUICK else 6)
rows = []
for sp in ("starling", "rook / crow"):
    for vh in (14.0, 20.0, 28.0):
        for k in KEYS:
            res = [pf.herd(ACS[k], UNIT, FARM, pf.Flock.of(sp, seed=s), v_herd=vh, fold_fast=30.0 if k == "C" else None, t_max=180.0)
                   for s in SEEDS]
            ok = [r_ for r_ in res if r_["cleared"]]
            rows.append({"species": sp, "herding speed [m/s]": vh, "wing": NAME[k], "cleared [%]": 100 * len(ok) / len(res),
                         "median time [s]": np.median([r_["time_s"] for r_ in ok]) if ok else np.nan,
                         "mean energy [Wh]": np.mean([r_["energy_wh"] for r_ in res])})
HERD = pd.DataFrame(rows)
display(HERD.pivot_table(index=["species", "herding speed [m/s]"], columns="wing", values=["cleared [%]", "median time [s]"]).round(1))
ex = pf.herd(ACS["C"], UNIT, FARM, pf.Flock.of("starling", seed=0), v_herd=28.0, fold_fast=30.0)
fig, ax = plt.subplots(figsize=(8, 8))
x0, y0, x1, y1 = FARM.field_xy
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, color="#e2c35a"))
ax.add_patch(plt.Circle(FARM.safe_xy, FARM.safe_r, fill=False, color="#2e6b2e", lw=2))
c = np.array([b.mean(axis=0) for b in ex["birds"]])
ax.plot(c[:, 0], c[:, 1], "k", lw=2, label="the flock's centre")
ax.plot(ex["pos"][:, 0], ex["pos"][:, 1], color=COLOR["C"], lw=0.8, label="C")
ax.scatter(ex["birds"][-1][:, 0], ex["birds"][-1][:, 1], s=6, c="k")
ax.set_aspect("equal"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title(f"C herding starlings: {'cleared in %.0f s' % ex['time_s'] if ex['cleared'] else 'not cleared'}, {ex['energy_wh']:.1f} Wh")

## 12. The roof: start, landing, rest

- **Prop-hang** (`pf.prop_hang`): nose up, the thrust carrying the weight — the static thrust/weight, the hover and climb
  power, then the transition to wing-borne flight (full power from 3 m/s to 1.3 × stall).
- **Hand throw** (`pf.hand_throw`): the slowest throw from 1.8 m that flies away at full power without touching the ground.
- **Drop off the edge** (`pf.roof_drop`, the falcon's start): the height needed to level off at 1.2 × stall — with no
  propeller hang. C dropping tucked unfolds as it levels.
- **Resting in the wind** (`pf.perched_wind`): facing into the wind on its skids at 5°, the ridge speeding the wind up
  ×1.3 and gusts ×1.4; it lifts off when the lift exceeds the weight, slides when the drag exceeds friction (0.6) ×
  (weight − lift). Here the fold could pay: a stowed wing lifts much less.

In [ ]:
rows = {}
for k in KEYS:
    ph = pf.prop_hang(ACS[k], UNIT)
    rows[NAME[k]] = {"static T/W": ph["T/W"], "hover power [W]": ph["hover power [W]"], "4 s hang [Wh]": ph["hang energy [Wh]"],
                     "transition [s]": ph["transition time [s]"], "slowest hand throw [m/s]": pf.min_throw_speed(ACS[k], UNIT),
                     "roof drop: height needed [m]": pf.roof_drop(ACS[k], UNIT)["height needed [m]"],
                     "touchdown speed, wing-borne [m/s]": 1.1 * ACS[k].af(0).stall_speed()}
ROOF = pd.DataFrame(rows)
display(ROOF.round(2))
folds = (0.0, 30.0, 60.0, 70.0, STOW)
WIND = pd.DataFrame([pf.perched_wind({"variant": "C"}, f, MASS["C"]) for f in folds]).set_index("fold [deg]")
for k in "AB":
    WIND.loc[f"{k} (fixed)"] = pf.perched_wind({"variant": k}, 0.0, MASS[k])
display(WIND.round(2))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(folds, [pf.perched_wind({"variant": "C"}, f, MASS["C"])["holds to [m/s]"] for f in folds], "o-", color=COLOR["C"], label=NAME["C"])
for k in "AB":
    ax.axhline(WIND.loc[f"{k} (fixed)", "holds to [m/s]"], color=COLOR[k], ls="--", label=NAME[k])
for bf, v in ((3, 3.4), (4, 5.5), (5, 8.0)):
    ax.axhline(v, color="grey", lw=0.6, ls=":"); ax.text(1, v + 0.1, f"Beaufort {bf}", fontsize=7, color="grey")
ax.set(xlabel="fold on the roof [deg]", ylabel="mean wind it stays put in [m/s]", title="resting on the ridge"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

**Reading it.** All three hang on the propeller (thrust/weight 1.45–1.9, ~120–175 W for a few seconds, a fraction of a
watt-hour) and all three fly away from a gentle hand throw. Dropping off the edge like a falcon needs 11–16 m — more
than this farmhouse's 8.5 m ridge, so the falcon's start is out and the prop-hang is the way. **On the ridge the fold
pays**: spread, the wings lift the aircraft off in a ~5.5–6.5 m/s breeze (with the ridge's speed-up and gusts);
stowed at 85°, C's wing lifts under a third as much and it stays put to ~9.7 m/s — Beaufort 4 to 5.

## 13. Structure (Talos) and the hinge

**Design load factor.** 5 g for the turns and the pull-out (birds pull 2–3 g; the margin is for the gusts near the ground
and the controller), checked against Pratt's gust formula at the stoop speed (as 26): `Δn = K_g ρ V U a S / (2 W)` with a
7.5 m/s gust. The wing FEA is 26's model on each wing (solid-equivalent LW-PLA, the centre section clamped, the lift as a
pressure on the lower skins); C spread (its largest bending) with the hinge treated as continuous — the hinge's own load
path is the lug below.

In [ ]:
V_DIVE = max(s["V_max"] for s in STOOP.values())
U_GUST = 7.5
gust = {}
for k in KEYS:
    a_ = pf.aero({"variant": k}, 0.0)
    W = MASS[k] * pf.G
    mu = 2 * W / a_["S_ref"] / (pf.RHO * a_["c_ref"] * a_["CLa"] * pf.G)
    kg = 0.88 * mu / (5.3 + mu)
    gust[k] = 1 + kg * pf.RHO * V_DIVE * U_GUST * a_["CLa"] * a_["S_ref"] / (2 * W)
N_DESIGN = max([MISSION["n_struct"]] + list(gust.values()))
print(f"stoop speed {V_DIVE:.0f} m/s: gust load factors", {k: round(v, 2) for k, v in gust.items()}, f"-> design load factor {N_DESIGN:.1f}")
a_t = pf.aero({"variant": "C"}, TUCK)
print(f"C tucked: CLα {a_t['CLa']:.2f}/rad (spread {pf.aero({'variant': 'C'}, 0.0)['CLa']:.2f}) — the tuck takes "
      f"{(1 - a_t['CLa'] / pf.aero({'variant': 'C'}, 0.0)['CLa']) * 100:.0f} % off the gust load at the same speed")

LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35, density=0.6e-9,
                        yield_strength=12.0, source="assumed (09a); replace with coupon tests")


def lower_skins(step):
    """The wing's lower skins: of each pair of spline faces at the same place, the lower one."""
    info = talos.inspect_step(step, units="mm-N-MPa")
    sk = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e3]
    out = []
    for s in sk:
        twin = [o for o in sk if o is not s and abs(o.centroid[1] - s.centroid[1]) < 5 and abs(o.centroid[0] - s.centroid[0]) < 40]
        if twin and all(s.centroid[2] < o.centroid[2] for o in twin):
            out.append(s.tag)
    return out


ws = core.Workspace.create(ROOT / "workspace", name="PEREGRINE")
design = ws.add_design("peregrine", f"{design_file}:Peregrine")
FEA, WREV = {}, {}
for k in KEYS:
    rev = design.new_revision(part="wing", **pg.VARIANTS[k], note=f"wing {k} for the FEA")
    rev.generate()
    WREV[k] = rev
    yc = rev.params["fuselage_diameter"] / 2 + 5.0
    S_k = pf.aero({"variant": k}, 0.0)["S_ref"]

    def wing_model(rev, k=k, yc=yc, S_k=S_k):
        regions = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)), talos.Surfaces("lift", lower_skins(rev.step))]
        return talos.StructuralModel(rev.step, "mm-N-MPa", LW_PLA, regions, [talos.FixedSupport("root")],
                                     [talos.Pressure("lift", N_DESIGN * MASS[k] * pf.G / (S_k * 1e6))],
                                     talos.MeshSettings(element_size=12.0 if QUICK else 8.0), name="design_load")
    FEA[k] = rev.run_fea("design_load", wing_model, progress=False)
WING = pd.DataFrame({NAME[k]: {"tip deflection [mm]": FEA[k].metrics.get("max_displacement", np.nan) if FEA[k].ok else np.nan,
                               "max von Mises [MPa]": FEA[k].metrics.get("max_von_mises", np.nan) if FEA[k].ok else np.nan,
                               "safety factor (yield)": FEA[k].metrics.get("safety_factor_yield", np.nan) if FEA[k].ok else np.nan}
                     for k in KEYS})
display(WING.round(2))

In [ ]:
ev = FEA["C"]
if ev.ok and ev.tool_results:
    tviz.show(tviz.plot_results(ev.tool_results[-1], field="von_mises"))
else:
    print("no FEA result to plot (is CalculiX installed?)", ev.messages if hasattr(ev, "messages") else "")

**The hinge.** `pf.hinge_loads`: the outer panel's lift (the lattice's span load at the load factor) acts at its centre
of pressure; the vertical pin carries the bending moment about the hinge as a couple between its two bearings, spaced by
the local wing depth — so the bearing force is several times the panel's lift. The fold servo turns the panel against
its drag, the bearings' friction under that couple, and the panel's inertia for a fold in 0.4 s. A 2:1 linkage halves
the torque at the servo and its speed; the servo comes from `actuators.py` (stall torque ≥ 1.5 × the need).

In [ ]:
rows = []
for f in (0.0, 30.0, TUCK):
    for V_, n_ in ((15.0, 3.0), (30.0, N_DESIGN), (V_DIVE, 1.0)):
        rows.append(pf.hinge_loads({"variant": "C"}, f, MASS["C"], n=n_, V=V_))
HINGE = pd.DataFrame(rows).set_index(["fold [deg]", "V [m/s]", "n"])
display(HINGE.round(3))
Q_MAX = HINGE["servo torque [N m]"].max()
RATIO = 2.0
SERVO = act.select(Q_MAX / RATIO, sf=1.5, kind="servo")
SERVO_LOCK = act.select(Q_MAX / RATIO, sf=1.5, kind="servo", self_locking=True)
fold_rate = SERVO.no_load_rpm * 6 / RATIO * 0.5          # deg/s at the panel: half the no-load speed under load
print(f"largest fold torque {Q_MAX:.2f} N m -> {Q_MAX / RATIO:.2f} N m at the servo through {RATIO:.0f}:1: {SERVO.key} "
      f"({SERVO.mass_g:.0f} g, stall {SERVO.stall_Nm} N m), panel fold rate ~{fold_rate:.0f}°/s "
      f"({TUCK / fold_rate:.2f} s to tuck); self-locking option for the roof: {SERVO_LOCK.key} ({SERVO_LOCK.mass_g:.0f} g)")
F_BEAR = HINGE["bearing force [N]"].max()
print(f"largest bearing force {F_BEAR:.0f} N: pin shear {HINGE['pin shear [MPa]'].max():.0f} MPa (steel pin: >300), "
      f"lug bearing {HINGE['lug bearing stress [MPa]'].max():.1f} MPa (printed PETG/PLA: ~40-50)")
ACS["C"].fold_rate_deg_s = fold_rate

**The hinge lug** (`part="hinge"`): a printed plate around the pin bore with a root block bonded to the spar. Talos:
the root block's bonded face fixed, the largest bearing force pulling the bore along the span (the lug in tension, the
lower lug of the pair; the upper one is in compression), printed PLA/PETG (solid-equivalent, conservative values).

In [ ]:
PETG = talos.Material("printed PETG lug (solid-equivalent)", youngs_modulus=1800.0, poissons_ratio=0.38, density=1.27e-9,
                      yield_strength=40.0, source="assumed: printed PETG, 100 % infill around the bore; replace with coupon tests")
lug = design.new_revision(part="hinge", **pg.VARIANTS["C"], note="hinge lug")
lug.generate()
pp = lug.params
L_, d_, t_ = pp["lug_length"], pp["hinge_pin_diameter"], pp["lug_thickness"]


def lug_model(rev):
    r = d_ / 2 + 0.1
    regions = [talos.SurfacesInBox("root", (-30.0, -20.1, -1.0, 30.0, -19.9, 13.0)),
               talos.SurfacesInBox("bore", (-r - 0.05, L_ - r - 0.05, -0.1, r + 0.05, L_ + r + 0.05, t_ + 0.1))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", PETG, regions, [talos.FixedSupport("root")],
                                 [talos.Force("bore", fy=F_BEAR)], talos.MeshSettings(element_size=1.0 if QUICK else 0.7), name="bearing")


EV_LUG = lug.run_fea("bearing", lug_model, progress=False)
if EV_LUG.ok:
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in EV_LUG.metrics.items() if k in ("max_von_mises", "max_displacement", "safety_factor_yield")})
    if EV_LUG.tool_results:
        tviz.show(tviz.plot_results(EV_LUG.tool_results[-1], field="von_mises"))
else:
    print("lug FEA not run:", EV_LUG.messages)

## 14. Printing (Mellonia)

The hinge lug flat on its plate, and the motor fairing (MERLIN's tractor nose), with the generic PLA profile. The lug's
grams go into the BOM (section 16 recomputes it with the sized servo and the weighed lugs).

In [ ]:
ev_lug = lug.run_print("flat", GENERIC_PLA_0_2MM, mellonia.Orientation())
nose = design.new_revision(part="nose", **pg.VARIANTS["A"], note="motor fairing, printed")
nose.generate()
ev_nose = nose.run_print("upright", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90))
PRINT = pd.DataFrame({name: {m: e.metrics.get(m) for m in ("estimated_time", "filament_used_g", "filament_cost", "layer_count")}
                      for name, e in (("hinge lug", ev_lug), ("motor fairing", ev_nose)) if e.ok})
display(PRINT)
if ev_lug.ok:
    LUG_G = float(ev_lug.metrics["filament_used_g"])

## 15. The sortie and the movie

`pf.mission` flies the whole sortie with each wing (MERLIN's point-mass logic, still air): prop-hang off the ridge, the
transition, a patrol lap at the best-endurance speed, a climb and a stoop at the deer (C tucked), a low pass at full
power, the starling flock herded to the wood (C tucks 30° on straight runs), home, the prop-hang landing on the ridge.
`pf.render_movie` draws C's (MERLIN's movie pipeline, the Vegeta watermark) into `output/28_peregrine/`.

In [ ]:
EPS = {k: pf.mission(ACS[k], UNIT, FARM, pf.Flock.of(MISSION["flock"], seed=0), patrol_agl=MISSION["patrol_agl_m"],
                     stoop_h=MISSION["stoop_from_m"], deer_xy=MISSION["deer_xy"], tuck=TUCK if k == "C" else 0.0,
                     fold_fast=30.0 if k == "C" else None, battery_wh=MISSION["battery_wh"]) for k in KEYS}
SORTIE = pd.DataFrame({NAME[k]: {"flight time [s]": ep.t[-1], "energy [Wh]": ep.energy_wh[-1],
                                 "sorties per charge": MISSION["battery_wh"] * MISSION["usable_fraction"] / ep.energy_wh[-1],
                                 "max airspeed [m/s]": ep.V.max(), "events": " | ".join(ep.events[1:-1])} for k, ep in EPS.items()})
display(SORTIE)
display(EPS["C"].phase_table().round(2))
fig = pf.profile_figure(list(EPS.values()), colors=COLOR)

In [ ]:
movie, n = pf.render_movie(EPS["C"], FARM, {"variant": "C"}, OUT / "peregrine_sortie.mp4", seconds=6.0 if QUICK else 30.0, fps=20,
                           title="PEREGRINE")
print(movie, n, "frames")
Video(str(movie), embed=False, width=960)

# Part 3 — the verdict

## 16. Does folding pay?

Every number the wings were compared on, in one table — what each wing does (its best way: C may fold where that helps)
and what it costs. The BOM is recomputed with the servo section 13 chose and the lugs section 14 weighed. The engineering
cost (`pf.ENGINEERING_HOURS`, editable estimates of a lab's hours) adds the hinge's design and tests, the fold's
software and a fold-in-flight test campaign. Scores: each metric normalised over the three wings (best 1, worst 0),
weighted, summed. The weights are a judgement, so the sensitivity draws thousands of random weightings of five groups
(endurance, agility, dive, roof, cost) and counts how often each wing wins.

In [ ]:
BOMS, BOM_TABLE = budget(fold_servo=SERVO.key, lug_g=LUG_G)
COST = {k: BOMS[k]["EUR"].sum() for k in KEYS}
MASS_FINAL = {k: BOMS[k]["mass [g]"].sum() for k in KEYS}
ENG = pf.engineering_cost()
display(ENG)


def best(k, fn, better="high"):
    folds = ACS[k].folds if k == "C" else [0.0]
    vals = [fn(f) for f in folds]
    return max(vals) if better == "high" else min(vals)


herd_s = HERD[HERD.species == "starling"].groupby("wing")
M = pd.DataFrame({k: {
    "endurance [min]": pf.envelope(ACS[k], UNIT)["endurance_min"],
    "range [km]": pf.envelope(ACS[k], UNIT)["range_km"],
    "top speed [m/s]": best(k, lambda f: pf.envelope(ACS[k], UNIT, fold=f)["v_top"]),
    "tightest turn [m]": pf.envelope(ACS[k], UNIT)["r_min_inst"],
    "turn rate [deg/s]": pf.envelope(ACS[k], UNIT)["rate_max_deg_s"],
    "gap 3 m deep [m]": GAPS.loc["gap 3 m deep [m]", NAME[k]],
    "herding cleared [%]": herd_s["cleared [%]"].mean()[NAME[k]],
    "stoop time from 150 m [s]": min(s["time_s"] for (kk, t_, h0), s in STOOP.items() if kk == k and h0 == MISSION["stoop_high_m"]),
    "stoop top speed [m/s]": max(s["V_max"] for (kk, t_, h0), s in STOOP.items() if kk == k and h0 == MISSION["stoop_high_m"]),
    "roof wind it holds [m/s]": WIND.loc[STOW, "holds to [m/s]"] if k == "C" else WIND.loc[f"{k} (fixed)", "holds to [m/s]"],
    "prop-hang T/W": ROOF.loc["static T/W", NAME[k]],
    "energy per sortie [Wh]": EPS[k].energy_wh[-1],
    "mass [g]": MASS_FINAL[k],
    "parts cost [EUR]": COST[k],
    "parts [count]": int((BOMS[k]["mass [g]"] > 0).sum()),
    "engineering hours": ENG.loc["total hours", k]} for k in KEYS})
BETTER = {"endurance [min]": "high", "range [km]": "high", "top speed [m/s]": "high", "tightest turn [m]": "low",
          "turn rate [deg/s]": "high", "gap 3 m deep [m]": "low", "herding cleared [%]": "high", "stoop time from 150 m [s]": "low",
          "stoop top speed [m/s]": "high", "roof wind it holds [m/s]": "high", "prop-hang T/W": "high",
          "energy per sortie [Wh]": "low", "mass [g]": "low", "parts cost [EUR]": "low", "parts [count]": "low", "engineering hours": "low"}
GROUPS = {"endurance": ["endurance [min]", "range [km]", "energy per sortie [Wh]"],
          "agility": ["top speed [m/s]", "tightest turn [m]", "turn rate [deg/s]", "gap 3 m deep [m]", "herding cleared [%]"],
          "dive": ["stoop time from 150 m [s]", "stoop top speed [m/s]"],
          "roof": ["roof wind it holds [m/s]", "prop-hang T/W"],
          "cost": ["mass [g]", "parts cost [EUR]", "parts [count]", "engineering hours"]}
WEIGHTS = {m: 1.0 / len(v) for g, v in GROUPS.items() for m in v}          # every group weighs the same
NORM, SCORE = pf.decision_table(M, BETTER, WEIGHTS)
display(M.round(2).rename(columns=NAME))
display(SCORE.rename(index=NAME).round(3).to_frame("score (equal groups)"))
WINS = pf.weight_sensitivity(M, BETTER, n=4000 if QUICK else 20000, groups=GROUPS)
print("share of random weightings each wing wins:", {NAME[k]: f"{v * 100:.0f} %" for k, v in WINS.items()})

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [2, 1]})
im = ax[0].imshow(NORM.values.astype(float), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels([NAME[k] for k in KEYS], fontsize=8)
ax[0].set_yticks(range(len(NORM))); ax[0].set_yticklabels(NORM.index, fontsize=8)
for i in range(len(NORM)):
    for j, k in enumerate(KEYS):
        ax[0].text(j, i, f"{M.iloc[i, j]:.3g}", ha="center", va="center", fontsize=7)
ax[0].set_title("each metric: best (green) to worst (red)")
ax[1].bar([NAME[k] for k in KEYS], [WINS[k] * 100 for k in KEYS], color=[COLOR[k] for k in KEYS])
ax[1].set(ylabel="% of random weightings won", title="who wins when the weights change"); ax[1].tick_params(axis="x", labelsize=8)
fig.tight_layout()

In [ ]:
fastest_pest = max(v[2] for v in pf.BIRDS.values() if v[4] is None)
CHECK = pd.DataFrame({NAME[k]: {
    "≥ 45 min patrol": M.loc["endurance [min]", k] >= 45,
    "≥ 1.3 x the pests' escape speed": M.loc["top speed [m/s]", k] >= 1.3 * fastest_pest,
    "turn inside ~10 m": M.loc["tightest turn [m]", k] <= 10.0,
    "a 1.2 m gap in a tree line": M.loc["gap 3 m deep [m]", k] <= 1.2,
    "stoop from 60 m, out at 3 m": abs(STOOP[(k, 0.0, MISSION["stoop_from_m"])]["h_bottom"] - MISSION["pass_agl_m"]) < 0.5,
    "herd starlings off the crop": M.loc["herding cleared [%]", k] >= 90,
    "start from the ridge": bool(ROOF.loc["static T/W", NAME[k]] > 1.1),
    "stay on the ridge in Beaufort 4 (5.5 m/s)": M.loc["roof wind it holds [m/s]", k] >= 5.5} for k in KEYS})
display(CHECK.replace({True: "yes", False: "NO"}))

**What a folding wing must buy.** C pays its hinge in money (parts and engineering hours, spread over a fleet) and in
grams (endurance). The per-aircraft premium over A and over B against the fleet size, with the engineering amortised:

In [ ]:
N = np.array([1, 2, 5, 10, 20, 50, 100])
rate = pf.ENGINEERING_RATE_EUR_H
per_unit = {k: COST[k] + ENG.loc["total hours", k] * rate / N for k in KEYS}
fig, ax = plt.subplots(figsize=(9, 4))
for k in KEYS:
    ax.plot(N, per_unit[k], "o-", color=COLOR[k], label=NAME[k])
ax.set(xscale="log", xlabel="aircraft built", ylabel="EUR per aircraft (parts + engineering / N)", title="the cost of the fold, amortised")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
print("per-aircraft premium of C over A: ", {int(n): round(float(per_unit['C'][i] - per_unit['A'][i])) for i, n in enumerate(N)})
print("per-aircraft premium of C over B: ", {int(n): round(float(per_unit['C'][i] - per_unit['B'][i])) for i, n in enumerate(N)})

# what C wins and loses, against the better of A and B on each metric
rows = []
for m in M.index:
    hi = BETTER[m] == "high"
    other = max(M.loc[m, "A"], M.loc[m, "B"]) if hi else min(M.loc[m, "A"], M.loc[m, "B"])
    who = ("A" if M.loc[m, "A"] == other else "B")
    gain = (M.loc[m, "C"] - other) * (1 if hi else -1)
    rows.append({"metric": m, "C": M.loc[m, "C"], "best fixed wing": other, "which": who, "C better by": gain,
                 "C better?": "yes" if gain > 1e-9 else ("same" if abs(gain) <= 1e-9 else "no")})
VS = pd.DataFrame(rows).set_index("metric")
display(VS.round(2))
wins_c = list(VS.index[VS["C better?"] == "yes"])
print("C beats both fixed wings on:", wins_c if wins_c else "nothing")
print(f"verdict (equal-group weights): {NAME[SCORE.idxmax()]} scores best; C wins {WINS['C'] * 100:.0f} % of random weightings")

### How to read the verdict

- **What the fold buys.** Folding is a span-and-lift-slope switch, not a drag switch: a sweep tuck keeps the wing's
  area, so the stoop gets only marginally faster (the hidden skin is a few percent of the drag). C beats both fixed
  wings on the stoop only by a fraction of a metre per second. It wins clearly where *less lift* matters: resting on
  a windy roof (stowed, it stays put in ~50 % more wind than the best fixed wing) and the gust load in a dive (the
  tucked wing's lift slope is ~40 % lower — a structural benefit the score does not count).
- **What it costs.** About 95 g over A (210 g over B) — the hinge hardware and servos take ~8 min off A's endurance —
  about 35 EUR of parts over A, five more parts, and the lab hours (the hinge, the fold's control, a fold-in-flight test
  campaign: ~105 h more). For one prototype the fold costs more than the whole aircraft; spread over a fleet of 100 it is
  ~80 EUR per aircraft.
- **Against B.** The cheap way to "fast and agile" is the smaller fixed wing: it banks through the same gaps C folds
  through, dives as fast, herds as well, starts most easily (highest thrust/weight) and is the cheapest and lightest.
  With every group of metrics weighted equally B scores best, and it wins most random weightings; C wins only when
  roof rest in wind is weighted heavily.
- **The answer to the question.** For the farm job the folding wing **does not justify its engineering cost** — unless
  the drones must rest on exposed roofs in stronger wind (a self-locking worm servo then holds the stow without power).
  As a lab experiment C is sound: the lattice and the trim say it flies and stays controllable at every fold, the hinge
  lug and pin carry the design load with margin, and a true root hinge — the falcon-like idea of the plan — does not
  even fit (section 3); the hinge belongs at the "elbow".

The tables above, not this text, are the result: change the weights, the prices or the mission and re-run.

## 17. Export

What another notebook or a scenario needs, as plain numbers (`designs/data/peregrine_design.json`): the wings'
parameters, masses and costs, the fold polar, the stability, the drive, the envelopes, the hinge and its servo, the
FEA, the sortie, the decision table and the sensitivity.

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x


def frame(df):
    return json.loads(df.to_json(orient="split", default_handler=str))


DESIGN = {"source": "28_peregrine", "variants": {k: dict(pg.VARIANTS[k]) for k in KEYS},
          "params": {k: {n: plain(v) for n, v in pg.resolve({"variant": k}).items()} for k in KEYS},
          "mass_g": {k: plain(MASS_FINAL[k]) for k in KEYS}, "parts_cost_eur": {k: plain(COST[k]) for k in KEYS},
          "battery_x_m": {k: plain(BATTERY_X[k]) for k in KEYS}, "fold_polar": {k: frame(v) for k, v in POLAR.items()},
          "cfd_cd0": {str(k): plain(v) for k, v in CFD_CD0.items()}, "stability": frame(STAB), "drive": UNIT.name,
          "envelope": frame(ENV), "gaps": frame(GAPS), "roof": frame(ROOF), "perched_wind": frame(WIND.reset_index()),
          "hinge": frame(HINGE.reset_index()), "fold_servo": SERVO.key, "fold_ratio": RATIO, "fold_rate_deg_s": plain(fold_rate),
          "design_load_factor": plain(N_DESIGN), "wing_fea": frame(WING), "lug_fea": {k: plain(v) for k, v in EV_LUG.metrics.items()} if EV_LUG.ok else None,
          "lug_g": plain(LUG_G), "sortie": frame(SORTIE), "metrics": frame(M), "score": {k: plain(v) for k, v in SCORE.items()},
          "wins": {k: plain(v) for k, v in WINS.items()}, "mission": MISSION, "fold_tuck_deg": TUCK, "fold_stow_deg": STOW}
path = Path("designs/data/peregrine_design.json")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(DESIGN, indent=1, default=plain))
print("written", path)